## baseline physics model


In [ ]:
import pandas as pd
df = pd.read_csv('final_data_physics_baseline/physics_basedate.csv')
df = df.set_index('date')
df.drop(columns= 'Unnamed: 0', inplace= True)
df.head()

In [ ]:
df1 = df.copy()

In [ ]:
#for single column dataframes always use double brackets, single brackets will make it a series, or float type
# for adding two columns to a df either add them one by one or use two brackets

In [ ]:
from sklearn.linear_model import LinearRegression
def create_lag(df, lag_months):
    df['sca_lagged'] = df['sca'].shift(lag_months)
    df['dd_lagged'] = df['dd'].shift(lag_months)

    df['melt_proxy'] = df['sca_lagged']*df['dd_lagged']
    df_clean = df.dropna()

    return df_clean

In [ ]:
x = create_lag(df1, lag_months=5)
x.head()

X_train = x[['precipitation','melt_proxy']]
Y_train = x[['runoff']]
X_train.head(10)
Y_val = x[['runoff']]
X_val = x[['precipitation','melt_proxy']]

In [ ]:
X = df1.drop(columns= 'runoff')
Y = df1[['runoff']]

In [ ]:
x_train = X_train[:'2017-12-31']
x_val = X_train['2018-01-01':'2021-12-31']
x_test = X_train['2022-01-01':]

y_train = Y_train[:'2017-12-31']
y_val = Y_train['2018-01-01':'2021-12-31']
y_test = Y_train['2022-01-01':]

In [ ]:
y_train

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaler.fit(x_train)
# x_train = scaler.transform(x_train)
# x_val = scaler.transform(x_val)
# x_test = scaler.transform(x_test)

In [ ]:
import numpy as np
def calculate_nse(observed, predicted):
    mean_obs = np.mean(observed)
    numerator = np.sum((observed - predicted) ** 2)
    denominator = np.sum((observed - mean_obs) ** 2)
    return 1 - (numerator / denominator)


In [ ]:
lag_scaler = StandardScaler()
lag_scaler.fit(X_train)

x_train_scaled = scaler.transform(x_train)
x_val_scaled = scaler.transform(x_val)


In [ ]:
model = LinearRegression()
model.fit(x_train_scaled, y_train)

y_val_pred = model.predict(x_val_scaled)
baseline_nse = calculate_nse(y_val, y_val_pred)

In [ ]:
baseline_nse